# Section VI-C: Single-shot vs Dynamic Circuit OTEM
**Fig. 21 & Table I** — failing rate comparison and IBM Q execution overhead.

Experimental protocol:
1. OT1 = online test circuit (for mitigation)
2. OT2 = second test circuit (used as target algorithm)
3. Three circuits submitted per repetition (interleaved): Original → SOTEM → DOTEM

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # qatg package
sys.path.insert(0, os.path.abspath('.'))    # OTEM.py, Result_analyze.py

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from copy import deepcopy
from qiskit import QuantumCircuit, transpile, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_experiments.library import StandardRB
from qiskit.result import marginal_counts
from otem import OTEM

## 1. Connect

In [ ]:
service = QiskitRuntimeService(
    channel='ibm_cloud',
    instance='YOUR_IBMQ_INSTANCE_HERE',
    token='YOUR_IBMQ_TOKEN_HERE'
)
backend_name = 'ibm_kyoto'   # change to target device
backend = service.backend(backend_name)
nqb     = backend.num_qubits
print(f"Backend: {backend.name}  ({nqb} qubits)")

In [ ]:
shots = 10000
nqc   = 29
test_datas = [QuantumCircuit().from_qasm_file(f'test_circuits/test_circuit_{i}.qasm')
              for i in range(nqc)]
my_em  = OTEM(backend, test_datas)
sampler = Sampler(backend)
preprocess_qcs = my_em.build_preprocess_test()

job_pre = sampler.run(preprocess_qcs, shots=shots)
print('Preprocess job:', job_pre.job_id())
pre_result = job_pre.result()
_, _, _, test_id = my_em.qubit_and_online_test_selection_from_result(pre_result)

## 2. Build OT1 (mitigation) and OT2 (target algorithm)

In [ ]:
def gen_ot_parallel(test_datas, test_id, nqb):
    qc, layouts = QuantumCircuit(0), []
    for i in range(nqb):
        if test_id[i] != -1:
            q2 = test_datas[test_id[i]].copy(); q2.remove_final_measurements()
            qc = q2.tensor(qc); layouts.append(i)
    return qc, layouts

# OT1: used as the mitigation test
qc_ot1, layouts = gen_ot_parallel(test_datas, test_id, nqb)
n_lay = len(layouts)
print(f"Selected {n_lay} qubits: {layouts}")

# OT2: used as the quantum algorithm (another test circuit acting as an application)
# Use the same layout; can be replaced with any application circuit
qc_ot2, _ = gen_ot_parallel(test_datas, test_id, nqb)  # same selection for demonstration

## 3. Build Single-shot OTEM and Dynamic Circuit OTEM variants

In [ ]:
def em_exp_on_layouts(qc_alg, qc_ot, layouts, backend):
    """Single-shot OTEM circuit."""
    qc = qc_ot.copy()
    crt = ClassicalRegister(n_lay, name='testresult')
    qc.add_register(crt); qc.measure(range(n_lay), crt)
    for g in qc_alg:
        qc.append(g.operation, qargs=g.qubits)
    crm = ClassicalRegister(n_lay, name='measureresult')
    qc.add_register(crm); qc.measure(range(n_lay), crm)
    return transpile(qc, backend, initial_layout=layouts, optimization_level=0)

def gen_dynamic_otem(test_datas, test_id, qc_ot1, layouts, backend):
    """Dynamic circuit OTEM: run algorithm only if OT passes."""
    qc = qc_ot1.copy()
    crt = ClassicalRegister(n_lay, name='testresult')
    qc.add_register(crt); qc.measure(range(n_lay), crt)
    for j, qb_phys in enumerate(layouts):
        if test_id[qb_phys] == -1:
            continue
        q2 = test_datas[test_id[qb_phys]].copy(); q2.remove_final_measurements()
        with qc.if_test((crt[j], 0)):
            for g in q2:
                qc.append(g.operation, qargs=[qc.qregs[0][j]])
    crm = ClassicalRegister(n_lay, name='measureresult')
    qc.add_register(crm); qc.measure(range(n_lay), crm)
    return transpile(qc, backend, initial_layout=layouts, optimization_level=0)

def gen_ori_qc(qc, layouts, backend):
    qc_ori = qc.copy()
    crm = ClassicalRegister(n_lay, name='measureresult')
    qc_ori.add_register(crm); qc_ori.measure(range(n_lay), crm)
    return transpile(qc_ori, backend, initial_layout=layouts, optimization_level=0)

qc_ori   = gen_ori_qc(qc_ot2, layouts, backend)
qc_sotem = em_exp_on_layouts(qc_ot2, qc_ot1, layouts, backend)
qc_dotem = gen_dynamic_otem(test_datas, test_id, qc_ot1, layouts, backend)

# Interleaved order per paper: original → SOTEM → DOTEM
N_TARGETS = 5
all_qcs = [qc for _ in range(N_TARGETS) for qc in [qc_ori, qc_sotem, qc_dotem]]
print(f"Total circuits: {len(all_qcs)}")

## 4. Submit and collect results

In [ ]:
job_main = sampler.run(all_qcs, shots=shots)
print('Main job:', job_main.job_id())
results  = list(job_main.result())

## 5. Compute failing rates and plot Fig. 21

In [ ]:
def otem_marginal_1qb(data, idx):
    tr = data.testresult.get_bitstrings()
    mr = data.measureresult.get_bitstrings()
    c = {}
    for t, m in zip(tr, mr):
        if t[idx] == '0': c[m[idx]] = c.get(m[idx],0)+1
    return c

from qiskit.result import marginal_counts

ori_fail, sotem_fail, dotem_fail = [], [], []
for i in range(N_TARGETS):
    r_ori, r_sotem, r_dotem = results[3*i], results[3*i+1], results[3*i+2]
    for qi in range(n_lay):
        o_c  = marginal_counts(r_ori.data.measureresult.get_counts(), [qi])
        s_c  = otem_marginal_1qb(r_sotem.data, -(qi+1))
        d_c  = otem_marginal_1qb(r_dotem.data, -(qi+1))
        def fr(c): tot=sum(c.values()); return 1-c.get('0',0)/tot if tot else 1.
        ori_fail.append(fr(o_c)); sotem_fail.append(fr(s_c)); dotem_fail.append(fr(d_c))

plt.figure(figsize=(6,6))
lim = max(max(ori_fail), 1.0)
plt.plot([0,lim],[0,lim],'k--',lw=1)
plt.scatter(ori_fail, sotem_fail, color='tab:blue',  marker='o', label='SOTEM', s=40)
plt.scatter(ori_fail, dotem_fail, color='tab:orange', marker='D', label='DOTEM', s=40)
plt.xlabel('Original failing rate'); plt.ylabel('Error mitigated failing rate')
plt.title('Fig. 21: Single-shot vs Dynamic Circuit OTEM'); plt.legend()
plt.tight_layout()
plt.savefig('fig21_sotem_vs_dotem.pdf', bbox_inches='tight')
plt.show()

## 6. Table I — IBM Q Usage (seconds)

In [ ]:
# After running on each device, retrieve job usage via job.metrics()
# Example: usage_s = job_main.metrics().get('usage', {}).get('seconds', None)
# Populate the table below with measured values:

print("Table I: IBM Q Usage (seconds)")
print(f"{'Device':<20} {'w/o EM':>10} {'SOTEM':>10} {'DOTEM':>10}")
print('-' * 52)
table = {
    'ibm_nazca':      (None, None, None),
    'ibm_kyoto':      (None, None, None),
    'ibm_brisbane':   (None, None, None),
    'ibm_sherbrooke': (None, None, None),
    'ibm_brussels':   (None, None, None),
}
for dev, (wo, so, do) in table.items():
    print(f"{dev:<20} {str(wo):>10} {str(so):>10} {str(do):>10}")